Silver and goldlayer path parameters

In [0]:
dbutils.widgets.text("silver_path", "")
dbutils.widgets.text("gold_path", "")

ADLS Congiguration for data access

In [0]:
storage_account = "adlsstoragedevesh"
account_key = "********************************************"
storage_account_url = "abfss://bronzelayer@adlsstoragedevesh.dfs.core.windows.net/bronze/"

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    account_key
)

Data loading from silver layer

In [0]:
base_path = "abfss://silverlayer@adlsstoragedevesh.dfs.core.windows.net/"

customer_df = spark.read.format("delta").load(base_path + "customer")
product_df = spark.read.format("delta").load(base_path + "product")
reseller_df = spark.read.format("delta").load(base_path + "reseller")
sales_df = spark.read.format("delta").load(base_path + "sales")
sales_order_df = spark.read.format("delta").load(base_path + "sales_order")
territory_df = spark.read.format("delta").load(base_path + "sales_territory")
date_df = spark.read.format("delta").load(base_path + "date")

Dimension tables creation

In [0]:
dim_customer = customer_df.select(
    "CustomerKey",
    "Customer",
    "City",
    "State-Province",
    "Country-Region",
    "Postal_Code"
)

In [0]:
dim_product = product_df.select(
    "ProductKey",
    "Product",
    "Category",
    "Subcategory",
    "Color"
)


In [0]:
dim_reseller = reseller_df.select(
    "ResellerKey",
    "Reseller",
    "Business_Type",
    "City",
    "Country-Region"
)


In [0]:
dim_territory = territory_df.select(
    "SalesTerritoryKey",
    "Region",
    "Country",
    "Group"
)

In [0]:
dim_date = date_df.select(
    "DateKey",
    "Full_Date",
    "Month",
    "Fiscal_Quarter",
    "Fiscal_Year"
)


Fact table creation by casting currency columns from string to double and performing broadcast join

In [0]:
from pyspark.sql.functions import col, regexp_replace, broadcast


fact_base = sales_df \
    .withColumn("Sales_Amount", regexp_replace(col("Sales_Amount"), "[$,]", "").cast("double")) \
    .withColumn("Total_Product_Cost", regexp_replace(col("Total_Product_Cost"), "[$,]", "").cast("double")) \
    .withColumn("Order_Quantity", col("Order_Quantity").cast("int"))


product_sel = product_df.select("ProductKey", "Product", "Category")
customer_sel = customer_df.select("CustomerKey", "Customer")
reseller_sel = reseller_df.select("ResellerKey", "Reseller")
territory_sel = territory_df.select("SalesTerritoryKey", "Region")
date_sel = date_df.select("DateKey", "Full_Date", "Month", "Fiscal_Year")
sales_order_sel = sales_order_df.select("SalesOrderLineKey", "Channel")


fact_sales = fact_base \
    .join(broadcast(product_sel), "ProductKey", "left") \
    .join(broadcast(customer_sel), "CustomerKey", "left") \
    .join(broadcast(reseller_sel), "ResellerKey", "left") \
    .join(broadcast(territory_sel), "SalesTerritoryKey", "left") \
    .join(
        broadcast(date_sel),
        fact_base["OrderDateKey"] == date_sel["DateKey"],
        "left"
    ) \
    .join(broadcast(sales_order_sel), "SalesOrderLineKey", "left")

fact_sales = fact_sales.drop(date_sel["DateKey"])

Saving files to gold layer

In [0]:
gold_base = "abfss://goldlayer@adlsstoragedevesh.dfs.core.windows.net/"

fact_sales.write.format("delta") \
    .mode("overwrite") \
    .save(gold_base + "fact_sales")

Creating Temporary views from the files saved in gold layer to use in table creation

In [0]:
gold_base = "abfss://goldlayer@adlsstoragedevesh.dfs.core.windows.net/"

spark.read.format("delta").load(gold_base + "fact_sales").createOrReplaceTempView("fact_sales")
spark.read.format("delta").load(gold_base + "dim_customer").createOrReplaceTempView("dim_customer")
spark.read.format("delta").load(gold_base + "dim_product").createOrReplaceTempView("dim_product")
spark.read.format("delta").load(gold_base + "dim_reseller").createOrReplaceTempView("dim_reseller")
spark.read.format("delta").load(gold_base + "dim_territory").createOrReplaceTempView("dim_territory")
spark.read.format("delta").load(gold_base + "dim_date").createOrReplaceTempView("dim_date")

gold_db creation to save tables for access in Power BI

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS gold_db;

Tables creation

In [0]:
%sql
CREATE OR REPLACE TABLE gold_db.fact_sales AS SELECT * FROM fact_sales;
CREATE OR REPLACE TABLE gold_db.dim_customer AS SELECT * FROM dim_customer;
CREATE OR REPLACE TABLE gold_db.dim_product AS SELECT * FROM dim_product;
CREATE OR REPLACE TABLE gold_db.dim_reseller AS SELECT * FROM dim_reseller;
CREATE OR REPLACE TABLE gold_db.dim_territory AS SELECT * FROM dim_territory;
CREATE OR REPLACE TABLE gold_db.dim_date AS SELECT * FROM dim_date;

num_affected_rows,num_inserted_rows
